# TODO

## Purpose: 

{TODO}

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, gzip, pickle 

## Literals


In [2]:
rmats_data_path = "/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/"

rmats_file_column_subset = ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE", "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2", "IncLevel1", "IncLevel2"]

cell_lines = ["K562", "HepG2"]

exon_ordering = {
    "+": {
        "1": "upstreamES", 
        "2": "upstreamEE",
        "3": "exonStart_0base", 
        "4": "exonEnd", 
        "5": "downstreamES", 
        "6": "downstreamEE"
    }, 
    "-": {
        "1": "downstreamEE", 
        "2": "downstreamES",
        "3": "exonEnd",
        "4": "exonStart_0base", 
        "5": "upstreamEE", 
        "6": "upstreamES"
    }
}

#### Get the RBPs per Cell Line that we are going to look at

In [3]:
selected_rbps = {}

control_associations = {}

for cell_line in cell_lines: 
    
    file = glob.glob("../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/{}*associations*".format(cell_line))
    assert len(file)==1
    
    tmp_df = pd.read_csv(file[0], sep="\t")
    
    selected_rbps[cell_line] = tmp_df["RBP KD"].to_list()
    
    control_associations[cell_line] = tmp_df 

## Load and Subset `rMATS Skipped Exon` Files

In [4]:
# dictionary with key as cell line and value is all the rMATS 
# files for that cell line concatted after subsetting for relevant columns 
rmats_cell_line_concat = {}

# for each cell line 
for cell_line in cell_lines: 
    cell_line 
    
    tmp_cell_line_rmats_df = []
    
    # for every SE file 
    for file in sorted(
        glob.glob("{}/*{}*/SE.*".format(rmats_data_path, cell_line), recursive=True)
    ): 

        # get rbp from the file path
        rbp = file.split("/")[-2].split("-")[0]
        # paranoia check: the cell line should be the same as what's labelled on the folder 
        assert file.split("/")[-2].split("-")[2] == cell_line
        
        # select only for RBPs we are looking at and for polyA mRNA samples
        if "-Transfection-" not in file and rbp in selected_rbps[cell_line]: 

            # read rMATS file and subset for relevant columns
            tmp_df = pd.read_csv(file, sep="\t")[rmats_file_column_subset]
            # add RBP KD column 
            tmp_df["RBP_KD_Target"] = rbp 
            
            # add df to list of dfs to be concatted 
            tmp_cell_line_rmats_df.append(tmp_df)
    
    # set dictionary key as cell line and value as concatted dfs 
    rmats_cell_line_concat[cell_line] = pd.concat(tmp_cell_line_rmats_df)

'K562'

'HepG2'

In [7]:
for cell_line in rmats_cell_line_concat: 
    rmats_cell_line_concat[cell_line].index.size

    rmats_cell_line_concat[cell_line].head()

10099472

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,RBP_KD_Target,PSI_KD_1,PSI_KD_2,PSI_CTRL_1,PSI_CTRL_2,Counts_KD_1,Counts_KD_2,Counts_CTRL_1,Counts_CTRL_2
0,chrX,+,156022698,156022834,156022313,156022459,156023011,156023209,AARS,0.833,0.579,0.929,1.0,11,15,27,31
1,chrX,-,155492356,155492495,155490114,155491666,155506897,155507134,AARS,1.0,1.0,0.941,1.0,20,45,33,27
2,chrX,-,155545095,155545277,155524270,155524632,155612791,155612877,AARS,1.0,1.0,1.0,0.93,111,132,124,165
3,chrX,+,155072326,155072343,155071419,155071650,155073376,155073431,AARS,0.81,1.0,1.0,1.0,24,21,18,13
4,chrX,+,155072326,155072343,155071419,155071650,155077169,155077283,AARS,1.0,0.631,1.0,1.0,8,9,3,2


9891471

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,RBP_KD_Target,PSI_KD_1,PSI_KD_2,PSI_CTRL_1,PSI_CTRL_2,Counts_KD_1,Counts_KD_2,Counts_CTRL_1,Counts_CTRL_2
0,chrY,+,14719458,14719518,14622008,14622591,14723116,14723269,AARS,0.651,0.509,0.533,0.609,16,24,17,14
1,chrY,+,14748618,14748729,14723116,14723269,14824187,14824373,AARS,0.059,0.0,0.0,0.0,9,7,3,9
2,chrY,+,14751680,14751802,14723116,14723269,14824187,14824373,AARS,0.0,0.067,0.0,0.0,8,8,3,9
3,chrY,+,12909359,12909407,12907536,12907594,12911838,12911968,AARS,1.0,0.996,1.0,1.0,374,408,231,210
4,chrY,+,12911838,12911968,12909359,12909407,12912726,12912882,AARS,1.0,0.994,1.0,0.994,521,668,425,337


## Separate `PSI` and `Total Counts` Into Separate Columns

In [6]:
for cell_line in rmats_cell_line_concat: 
        
    tmp_df = rmats_cell_line_concat[cell_line]

    tmp_df["PSI_KD_1"] = tmp_df["IncLevel1"].str.split(",").str[0]
    tmp_df["PSI_KD_2"] = tmp_df["IncLevel1"].str.split(",").str[1]

    tmp_df["PSI_CTRL_1"] = tmp_df["IncLevel2"].str.split(",").str[0]
    tmp_df["PSI_CTRL_2"] = tmp_df["IncLevel2"].str.split(",").str[1]

    tmp_df = tmp_df.drop(columns = ["IncLevel1", "IncLevel2"])

    tmp_df["Counts_KD_1"] = (tmp_df["IJC_SAMPLE_1"].str.split(",").str[0]).astype("int64") + (tmp_df["SJC_SAMPLE_1"].str.split(",").str[0]).astype("int64")
    tmp_df["Counts_KD_2"] = (tmp_df["IJC_SAMPLE_1"].str.split(",").str[1]).astype("int64") + (tmp_df["SJC_SAMPLE_1"].str.split(",").str[1]).astype("int64")
    tmp_df["Counts_CTRL_1"] = (tmp_df["IJC_SAMPLE_2"].str.split(",").str[0]).astype("int64") + (tmp_df["SJC_SAMPLE_2"].str.split(",").str[0]).astype("int64")
    tmp_df["Counts_CTRL_2"] = (tmp_df["IJC_SAMPLE_2"].str.split(",").str[1]).astype("int64") + (tmp_df["SJC_SAMPLE_2"].str.split(",").str[1]).astype("int64")

    tmp_df = tmp_df.drop(columns=["IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2"])

    rmats_cell_line_concat[cell_line] = tmp_df


In [8]:
tmp_df.head()

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,RBP_KD_Target,PSI_KD_1,PSI_KD_2,PSI_CTRL_1,PSI_CTRL_2,Counts_KD_1,Counts_KD_2,Counts_CTRL_1,Counts_CTRL_2
0,chrY,+,14719458,14719518,14622008,14622591,14723116,14723269,AARS,0.651,0.509,0.533,0.609,16,24,17,14
1,chrY,+,14748618,14748729,14723116,14723269,14824187,14824373,AARS,0.059,0.0,0.0,0.0,9,7,3,9
2,chrY,+,14751680,14751802,14723116,14723269,14824187,14824373,AARS,0.0,0.067,0.0,0.0,8,8,3,9
3,chrY,+,12909359,12909407,12907536,12907594,12911838,12911968,AARS,1.0,0.996,1.0,1.0,374,408,231,210
4,chrY,+,12911838,12911968,12909359,12909407,12912726,12912882,AARS,1.0,0.994,1.0,0.994,521,668,425,337


## Load `Splice Junction to # RBP Peaks` Data

In [9]:
junction_to_num_peaks = {}

with gzip.GzipFile("../../5_assign_eCLIP_to_splice_junctions/output/splice_junction_rbp_num_peaks/all_RBP_peaks_num_per_splice_junction.pkl.gz", 'rb') as in_file: 
    junction_to_num_peaks = pickle.load(in_file)


In [15]:
junction_to_num_peaks['HepG2'].keys()

dict_keys([50, 100, 500, 1000, 2000, 5000, 10000])

## Create Input Data for ML Model by Starting w/ Creating # Peaks per RBP per Sample

#### Pseudocode of Algorithm

In [ ]:
for cell line: 
    for distance threshold: 
                
        for each row in concat rmats dict: 
            get the correct orientation of which column corresponds to 1-6 in our paradigm 
            
            for each sample in row: 
                if not "nan": 
                    
                    create unique_id: should be chr, strand, all 6 splice junction coordinates for that event, sample name, and RBP KD (if applicable)
                    (for controls, use control accession lookup table to convert to ENCSR ID and do not include RBP KD)
                    
                    if unique id not in dictionary: 
                        
                        save "chr", inclevel (target), total counts,

                        for each position (1-6): 
                            
                            create lookup string: chr_splice-jnction-coordinate_strand
                            
                            for each rbp that we are looking at in that cell line: 
                                use lookup string to save number of peaks total 
                
                                saving should be done using dictionary structure such as: 
                                    {
                                        unique_id: {
                                            total counts: value
                                            RBP_1_binding: value 
                                        }
                                    }

In [ ]:
for cell_line in junction_to_num_peaks: 
    for threshold in junction_to_num_peaks[cell_line]: 
        
        